# New QE — Kaggle training setup

## Goal
Four separate scratch models at QP22/27/32/37, Charbonnier-only, 300k successful updates each. Best checkpoint uses maximum validation PSNR-Y. This notebook defaults to setup and smoke only; main training is opt-in.

Attach public dataset **anginhok/enhance-for-human-mfqe-vvc-trainval-qp22-37**. Enable a GPU and Internet for setup. Validation uses the first 30 frames of test18; this bundle excludes full test18.

## Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path("/kaggle/working/Enhance-for-Human")
DATASET = Path("/kaggle/input/enhance-for-human-mfqe-vvc-trainval-qp22-37")
RUN_GPU_SMOKE = True
RUN_MAIN = False

if not PROJECT.exists():
    subprocess.run(["git", "clone", "https://github.com/dangdinh17/Enhance-for-Human.git", str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "tools/researchctl"], check=True)

## Steps

### 1. Verify dataset and attach read-only LMDBs
SHA-256 verification reads approximately 73.70 GiB on the first run. Subsequent calls reuse the receipt while local file identities are unchanged; LMDBs remain on Kaggle Input.

In [ ]:
subprocess.run([sys.executable, "-B", "scripts/bootstrap_kaggle.py", "--dataset-root", str(DATASET)], check=True)

CAMPAIGN = "newqe_kaggle_charb_300k_v1"
def ctl(profile, *args):
    return subprocess.run([sys.executable, "-m", "researchctl.cli", "--profile", profile, *args], check=True)

ctl("smoke", "validate")
ctl("smoke", "plan", "--campaign", CAMPAIGN)
ctl("smoke", "preflight", "--campaign", CAMPAIGN)

### 2. Bounded GPU smoke and resume
Two updates per QP, eight validation samples per QP; diagnostic results only. The smoke adapter stops after update 1, reloads all state, then reaches update 2.

In [ ]:
if RUN_GPU_SMOKE:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU before running GPU smoke")
    ctl("smoke", "status", "--campaign", CAMPAIGN)
    ctl("smoke", "run", "--campaign", CAMPAIGN, "--until", "smoke")

## Checks

### 3. Optional main training
Main uses all 540 validation samples and requires its own Kaggle-profile smoke and explicit approval. Set RUN_MAIN=True only when ready to train. Preserve the run directory and exact code/config for resume; do not replace existing weights with a new scratch run.

In [ ]:
if RUN_MAIN:
    ctl("kaggle", "validate")
    ctl("kaggle", "plan", "--campaign", CAMPAIGN)
    ctl("kaggle", "preflight", "--campaign", CAMPAIGN)
    ctl("kaggle", "run", "--campaign", CAMPAIGN, "--until", "smoke")
    ctl("kaggle", "approve", "--campaign", CAMPAIGN, "--stage", "main")
    ctl("kaggle", "run", "--campaign", CAMPAIGN, "--until", "main")

## Next Steps
Save/download **runs/research/** before ending a session. On another session restore it to the same project path, use the same repository commit and dependency versions, rerun bootstrap, then use researchctl resume with the kaggle profile. Approval may need renewal if environment/provenance changes. Logs and best/last checkpoints stay under runs; main training is not executed by this release task.

In [ ]:
import shutil
RUNS = PROJECT / "runs"
if RUNS.exists():
    archive = shutil.make_archive("/kaggle/working/newqe-runs-resume", "gztar", root_dir=PROJECT, base_dir="runs")
    print("Resume archive:", archive)